# Snowflake Cortex AI SQL: Extracting Insights from Multimodal Customer Service Data

This notebook demonstrates how to build a comprehensive customer service analytics system using **Snowflake Cortex AI functions**. You'll process audio recordings, chat logs, support tickets, and PDF documents - all using pure SQL.

### What You'll Learn
| AI Function | Purpose | Input | Output |
|-------------|---------|-------|--------|
| `AI_TRANSCRIBE` | Speech-to-text | Audio files | Text with speaker segments |
| `AI_TRANSLATE` | Language translation | Any text | English text |
| `AI_SENTIMENT` | Emotion detection | Text | positive/negative/neutral/mixed |
| `AI_CLASSIFY` | Categorization | Text + categories | Best matching category |
| `AI_COMPLETE` | LLM generation | Prompt | Generated text |
| `AI_PARSE_DOCUMENT` | Document extraction | PDF files | Structured content |
| `AI_EXTRACT` | Field extraction | Text + schema | Structured JSON |

### Prerequisites
- Run `setup.sql` before starting this notebook
- Ensure your account has Cortex AI functions enabled
- Use a warehouse with sufficient compute (MEDIUM or larger recommended)

## Step 0: Explore the Sample Data

Before processing, let's understand what data we're working with. The setup script created:
- **Audio files**: Customer service call recordings in the `@CUSTOMER_CALLS` stage
- **PDF documents**: Company documents in the `@COMPANY_DOCUMENTS` stage  
- **Chat logs**: Customer chat transcripts with agent classifications
- **Support tickets**: Formal support tickets linked to chats

In [ ]:
-- View available audio files
SELECT * FROM DATA.audio_file_list;

In [ ]:
-- Preview chat logs structure
SELECT chat_id, customer_name, self_reported_category, self_reported_sentiment, resolution_status
FROM CHAT_LOGS LIMIT 5;

In [ ]:
-- Preview support tickets
SELECT ticket_id, ticket_number, subject, self_reported_category, priority, product_affected
FROM SUPPORT_TICKETS LIMIT 5;

---
## Part 1: Audio Processing Pipeline

### The Complete Pipeline (Production Approach)

This query chains **6 AI functions** together in a single statement to process customer calls:

```
Audio File → AI_TRANSCRIBE → AI_TRANSLATE → AI_SENTIMENT → AI_CLASSIFY → AI_COMPLETE → Results Table
```

**Why chain them?** Each function builds on the previous output:
1. Can't analyze sentiment until we have text (transcription)
2. Can't classify until we have English text (translation)
3. Summaries need the full translated conversation

⏱️ **Expected runtime**: ~30-60 seconds per audio file (depends on length)

In [ ]:
INSERT INTO transcription_results (
    stage_location,
    file_name,
    timestamp_granularity,
    audio_duration,
    segments,
    raw_response,
    translated_text,
    sentiment_label,
    call_category,
    call_summary,
    transcription_completed_at
)
WITH transcriptions AS (
    SELECT 
        file_name,
        AI_TRANSCRIBE(
            TO_FILE('@MULTIMODAL_CUSTOMER_SERVICE.DATA.Customer_Calls/' || file_name),
            OBJECT_CONSTRUCT('timestamp_granularity', 'speaker')
        ) AS trans_result
    FROM data.audio_file_list
),
with_transcripts AS (
    SELECT
        file_name,
        trans_result,
        ARRAY_TO_STRING(
            ARRAY_AGG(seg.value:text::VARCHAR) WITHIN GROUP (ORDER BY seg.index),
            ' '
        ) AS full_transcript
    FROM transcriptions,
         TABLE(FLATTEN(input => trans_result:segments)) seg
    GROUP BY file_name, trans_result
),
with_translation AS (
    SELECT
        file_name,
        trans_result,
        full_transcript,
        AI_TRANSLATE(full_transcript,'' ,'en') AS translated_transcript
    FROM with_transcripts
)
SELECT
    '@MULTIMODAL_CUSTOMER_SERVICE.DATA.Customer_Calls/' || file_name AS stage_location,
    file_name,
    'speaker' AS timestamp_granularity,
    trans_result:audio_duration::FLOAT AS audio_duration,
    trans_result:segments AS segments,
    trans_result AS raw_response,
    translated_transcript AS translated_text,
    AI_SENTIMENT(translated_transcript):categories[0]:sentiment::VARCHAR AS sentiment_label,
    AI_CLASSIFY(
        translated_transcript,
        [
            OBJECT_CONSTRUCT('label', 'Fraud & Security Issues', 
                             'description', 'Unauthorized transactions, identity theft, account freezes, fraudulent charges'),
            OBJECT_CONSTRUCT('label', 'Technical & System Errors',
                             'description', 'Auto-pay failures, system glitches, login problems, display issues, platform malfunctions'),
            OBJECT_CONSTRUCT('label', 'Payment & Transaction Problems',
                             'description', 'Duplicate charges, failed payments, processing errors, fee disputes, rate increases'),
            OBJECT_CONSTRUCT('label', 'Account Changes & Modifications',
                             'description', 'Investment reallocation, policy modifications, coverage changes, fund transfers, limit adjustments'),
            OBJECT_CONSTRUCT('label', 'General Inquiries & Information Requests',
                             'description', 'Status checks, documentation requests, simple questions, coverage verification, timeline questions')
        ],
        OBJECT_CONSTRUCT('task_description', 'Classify customer service calls into issue categories based on the main problem discussed')
    ):labels[0]::VARCHAR AS call_category,
    AI_COMPLETE(
        'claude-sonnet-4-5',
        CONCAT('Summarize this call in 50 words: ', translated_transcript)
    ) AS call_summary,
    CURRENT_TIMESTAMP() AS transcription_completed_at
FROM with_translation;

---
### Breaking Down Each AI Function

The following cells demonstrate each function individually so you can understand what's happening at each step.

## AI_TRANSCRIBE

Converts audio files to text with speaker diarization (identifying who said what).

**Key parameters:**
- `TO_FILE()`: Points to the audio file in a stage
- `timestamp_granularity: 'speaker'`: Groups text by speaker turns

**Output structure:**
```json
{
  "audio_duration": 45.2,
  "segments": [
    {"speaker": "SPEAKER_00", "text": "Hello, how can I help?", "start": 0.0, "end": 2.1},
    {"speaker": "SPEAKER_01", "text": "I have a billing issue...", "start": 2.5, "end": 8.3}
  ]
}
```

In [ ]:
CREATE OR REPLACE TEMPORARY TABLE temp_transcriptions AS
SELECT 
    file_name,
    AI_TRANSCRIBE(
        TO_FILE('@MULTIMODAL_CUSTOMER_SERVICE.DATA.Customer_Calls/' || file_name),
        OBJECT_CONSTRUCT('timestamp_granularity', 'speaker')
    ) AS trans_result
FROM data.audio_file_list;

SELECT * FROM temp_transcriptions LIMIT 5;

### Flattening the Transcript Segments

The transcription returns an array of segments. We use `FLATTEN()` to explode this array and `ARRAY_AGG()` to combine all text into one conversation string.

**Why?** Downstream AI functions (sentiment, classification) work better on complete text rather than fragments.

In [ ]:
CREATE OR REPLACE TEMPORARY TABLE temp_with_transcripts AS
SELECT
    file_name,
    trans_result,
    ARRAY_TO_STRING(
        ARRAY_AGG(seg.value:text::VARCHAR) WITHIN GROUP (ORDER BY seg.index),
        ' '
    ) AS full_transcript
FROM temp_transcriptions,
     TABLE(FLATTEN(input => trans_result:segments)) seg
GROUP BY file_name, trans_result;

SELECT * FROM temp_with_transcripts LIMIT 5;

## AI_TRANSLATE

Automatically detects the source language and translates to English (or any target language).

**Syntax:** `AI_TRANSLATE(text, source_language, target_language)`
- Use empty string `''` for source to auto-detect
- Use ISO language codes: `'en'`, `'es'`, `'fr'`, `'de'`, etc.

**Use case:** Global customer service teams can analyze all calls in a single language regardless of the original conversation language.

In [ ]:
CREATE OR REPLACE TEMPORARY TABLE temp_with_translation AS
SELECT
    file_name,
    trans_result,
    full_transcript,
    AI_TRANSLATE(full_transcript, '', 'en') AS translated_transcript
FROM temp_with_transcripts;

SELECT * FROM temp_with_translation LIMIT 5;

## AI_SENTIMENT

Analyzes the emotional tone of text and returns a sentiment classification.

**Output structure:**
```json
{
  "categories": [
    {"sentiment": "negative", "score": 0.85},
    {"sentiment": "neutral", "score": 0.10},
    {"sentiment": "positive", "score": 0.05}
  ]
}
```

**Possible values:** `positive`, `negative`, `neutral`, `mixed`, `very_negative`

**Business use:** Identify frustrated customers for escalation, track sentiment trends over time.

In [ ]:
CREATE OR REPLACE TEMPORARY TABLE temp_with_sentiment AS
SELECT
    file_name,
    trans_result,
    full_transcript,
    translated_transcript,
    AI_SENTIMENT(translated_transcript):categories[0]:sentiment::VARCHAR AS sentiment_label
FROM temp_with_translation;

SELECT * FROM temp_with_sentiment LIMIT 5;

## AI_CLASSIFY

Categorizes text into one of your custom-defined categories. This is **zero-shot classification** - no training required!

**Key features:**
- Define your own categories with labels and descriptions
- Descriptions help the AI understand what belongs in each category
- Returns confidence scores for each category

**Our 5 categories:**
1. Fraud & Security Issues
2. Technical & System Errors  
3. Payment & Transaction Problems
4. Account Changes & Modifications
5. General Inquiries & Information Requests

In [ ]:
CREATE OR REPLACE TEMPORARY TABLE temp_with_classification AS
SELECT
    file_name,
    trans_result,
    full_transcript,
    translated_transcript,
    sentiment_label,
    AI_CLASSIFY(
        translated_transcript,
        [
            OBJECT_CONSTRUCT('label', 'Fraud & Security Issues', 
                             'description', 'Unauthorized transactions, identity theft, account freezes, fraudulent charges'),
            OBJECT_CONSTRUCT('label', 'Technical & System Errors',
                             'description', 'Auto-pay failures, system glitches, login problems, display issues, platform malfunctions'),
            OBJECT_CONSTRUCT('label', 'Payment & Transaction Problems',
                             'description', 'Duplicate charges, failed payments, processing errors, fee disputes, rate increases'),
            OBJECT_CONSTRUCT('label', 'Account Changes & Modifications',
                             'description', 'Investment reallocation, policy modifications, coverage changes, fund transfers, limit adjustments'),
            OBJECT_CONSTRUCT('label', 'General Inquiries & Information Requests',
                             'description', 'Status checks, documentation requests, simple questions, coverage verification, timeline questions')
        ],
        OBJECT_CONSTRUCT('task_description', 'Classify customer service calls into issue categories based on the main problem discussed')
    ):labels[0]::VARCHAR AS call_category
FROM temp_with_sentiment;


SELECT * FROM temp_with_classification LIMIT 5;

## AI_COMPLETE

The most flexible function - runs any prompt through an LLM. Here we use it for summarization.

**Syntax:** `AI_COMPLETE(model_name, prompt)`

**Available models:**
- `claude-sonnet-4-5` - High quality, balanced speed/cost (used here)
- `llama3.1-70b` - Open source alternative
- `mistral-large2` - Fast, cost-effective
- See docs for full list

**Pro tip:** Be specific in your prompt about output format and length constraints.

In [ ]:
CREATE OR REPLACE TEMPORARY TABLE temp_with_summary AS
SELECT
    file_name,
    trans_result,
    full_transcript,
    translated_transcript,
    sentiment_label,
    call_category,
    AI_COMPLETE(
        'claude-sonnet-4-5',
        CONCAT('Summarize this call in 50 words: ', translated_transcript)
    ) AS call_summary
FROM temp_with_classification;

SELECT * FROM temp_with_summary LIMIT 5;

## Combine All Results

Now we insert all the processed data into our permanent results table. This is the same data the "complex query" at the top would generate, but here we built it step-by-step.

In [ ]:
INSERT INTO transcription_results (
    stage_location,
    file_name,
    timestamp_granularity,
    audio_duration,
    segments,
    raw_response,
    translated_text,
    sentiment_label,
    call_category,
    call_summary,
    transcription_completed_at
)
SELECT
    '@MULTIMODAL_CUSTOMER_SERVICE.DATA.Customer_Calls/' || file_name AS stage_location,
    file_name,
    'speaker' AS timestamp_granularity,
    trans_result:audio_duration::FLOAT AS audio_duration,
    trans_result:segments AS segments,
    trans_result AS raw_response,
    translated_transcript AS translated_text,
    sentiment_label,
    call_category,
    call_summary,
    CURRENT_TIMESTAMP() AS transcription_completed_at
FROM temp_with_summary;

## Clean Up Temporary Tables

Remove the intermediate tables we created during the step-by-step demonstration.

In [ ]:
DROP TABLE IF EXISTS temp_transcriptions;
DROP TABLE IF EXISTS temp_with_transcripts;
DROP TABLE IF EXISTS temp_with_translation;
DROP TABLE IF EXISTS temp_with_sentiment;
DROP TABLE IF EXISTS temp_with_classification;
DROP TABLE IF EXISTS temp_with_summary;

## View Final Results

Query the results table to see all processed calls with their transcriptions, translations, sentiments, categories, and AI-generated summaries.

In [ ]:
SELECT * FROM transcription_results;

### Quick Analysis of Results

Let's see the distribution of categories and sentiments across our processed calls.

In [ ]:
SELECT 
    call_category,
    sentiment_label,
    COUNT(*) as call_count,
    ROUND(AVG(audio_duration), 1) as avg_duration_seconds
FROM transcription_results
GROUP BY call_category, sentiment_label
ORDER BY call_count DESC;

---
## Part 2: Document Processing with AI_PARSE_DOCUMENT

Extract structured content from PDF documents. This function:
- Extracts text while preserving layout
- Identifies headers, paragraphs, tables
- Can split by page for large documents

**Parameters:**
- `mode: 'LAYOUT'` - Preserves document structure
- `page_split: TRUE` - Returns content page-by-page

**Use cases:** Policy documents, contracts, invoices, manuals

In [ ]:
CREATE TABLE IF NOT EXISTS parsed_documents_raw AS
SELECT 
    RELATIVE_PATH AS file_name,
    '@COMPANY_DOCUMENTS/' || RELATIVE_PATH AS file_path,
    SIZE AS file_size_bytes,
    LAST_MODIFIED,
    AI_PARSE_DOCUMENT(
        TO_FILE('@COMPANY_DOCUMENTS', RELATIVE_PATH),
        {'mode': 'LAYOUT', 'page_split': TRUE}
    ) AS parsed_result
FROM DIRECTORY(@COMPANY_DOCUMENTS);

Select * from parsed_documents_raw limit 10;

---
## Part 3: Validating Chat Logs with AI

Customer service agents manually classify chats, but humans make mistakes. Let's use AI to:
1. **Re-classify** conversations automatically
2. **Re-analyze** sentiment
3. **Extract** key information
4. **Flag** discrepancies between agent labels and AI analysis

This introduces `AI_EXTRACT` - a function that pulls structured fields from unstructured text based on a schema you define.

In [ ]:
CREATE OR REPLACE TABLE chat_validation_results AS
WITH conversation_prep AS (
    SELECT 
        c.chat_id,
        c.ticket_id,
        c.customer_name,
        c.agent_name,
        c.chat_timestamp,
        c.self_reported_category,
        c.self_reported_sentiment,
        c.resolution_status,
        ARRAY_TO_STRING(
            ARRAY_AGG(msg.value:message::VARCHAR) 
            WITHIN GROUP (ORDER BY msg.index), 
            ' '
        ) AS full_conversation
    FROM chat_logs c,
    LATERAL FLATTEN(input => PARSE_JSON(c.messages)) msg
    GROUP BY 
        c.chat_id, c.ticket_id, c.customer_name, c.agent_name,
        c.chat_timestamp, c.self_reported_category, 
        c.self_reported_sentiment, c.resolution_status
),
ai_analysis AS (
    SELECT 
        chat_id,
        ticket_id,
        customer_name,
        agent_name,
        chat_timestamp,
        self_reported_category,
        self_reported_sentiment,
        resolution_status,
        full_conversation,
        AI_CLASSIFY(
            full_conversation,
            ARRAY_CONSTRUCT('Technical Support', 'Bug Report', 'Feature Request', 'Billing', 'General Inquiry')
        ) AS ai_classify_object,
        AI_SENTIMENT(full_conversation) AS ai_sentiment_object,
        AI_EXTRACT(
            text => full_conversation,
            responseFormat => OBJECT_CONSTRUCT(
                'issue_description', 'What is the main issue or problem described?',
                'product_name', 'What product is mentioned?',
                'error_message', 'What error message or code is mentioned?',
                'resolution_provided', 'What solution or resolution was provided?',
                'customer_satisfaction_indicator', 'Does the customer seem satisfied?',
                'urgency_level', 'How urgent is this issue?'
            )
        ) AS extracted_features
    FROM conversation_prep
)
SELECT 
    chat_id,
    ticket_id,
    customer_name,
    agent_name,
    chat_timestamp,
    self_reported_category,
    self_reported_sentiment,
    resolution_status,
    full_conversation,
    ai_classify_object:labels[0]::VARCHAR AS ai_classified_category,
    ai_sentiment_object:categories[0]:sentiment::VARCHAR AS ai_sentiment_raw,
    CASE 
        WHEN ai_sentiment_object:categories[0]:sentiment::VARCHAR = 'positive' THEN 'Positive'
        WHEN ai_sentiment_object:categories[0]:sentiment::VARCHAR = 'neutral' THEN 'Neutral'
        WHEN ai_sentiment_object:categories[0]:sentiment::VARCHAR IN ('negative', 'very_negative') THEN 'Negative'
        WHEN ai_sentiment_object:categories[0]:sentiment::VARCHAR = 'mixed' THEN 'Mixed'
        ELSE 'Unknown'
    END AS ai_sentiment_normalized,
    extracted_features,
    CASE 
        WHEN self_reported_category != ai_classify_object:labels[0]::VARCHAR 
         AND ai_classify_object:labels[0]::VARCHAR IS NOT NULL THEN TRUE
        WHEN self_reported_sentiment != CASE 
                WHEN ai_sentiment_object:categories[0]:sentiment::VARCHAR = 'positive' THEN 'Positive'
                WHEN ai_sentiment_object:categories[0]:sentiment::VARCHAR = 'neutral' THEN 'Neutral'
                WHEN ai_sentiment_object:categories[0]:sentiment::VARCHAR IN ('negative', 'very_negative') THEN 'Negative'
                ELSE 'Unknown'
            END
         AND NOT (self_reported_sentiment = 'Frustrated' 
                  AND ai_sentiment_object:categories[0]:sentiment::VARCHAR IN ('negative', 'very_negative'))
         AND ai_sentiment_object:categories[0]:sentiment::VARCHAR IS NOT NULL THEN TRUE
        WHEN ai_sentiment_object:categories[0]:sentiment::VARCHAR = 'mixed' THEN TRUE
        ELSE FALSE
    END AS is_flagged,
    ARRAY_CONSTRUCT_COMPACT(
        CASE 
            WHEN self_reported_category != ai_classify_object:labels[0]::VARCHAR
             AND ai_classify_object:labels[0]::VARCHAR IS NOT NULL
            THEN 'category_mismatch'
        END,
        CASE 
            WHEN self_reported_sentiment != CASE 
                    WHEN ai_sentiment_object:categories[0]:sentiment::VARCHAR = 'positive' THEN 'Positive'
                    WHEN ai_sentiment_object:categories[0]:sentiment::VARCHAR = 'neutral' THEN 'Neutral'
                    WHEN ai_sentiment_object:categories[0]:sentiment::VARCHAR IN ('negative', 'very_negative') THEN 'Negative'
                    ELSE 'Unknown'
                END
             AND NOT (self_reported_sentiment = 'Frustrated' 
                      AND ai_sentiment_object:categories[0]:sentiment::VARCHAR IN ('negative', 'very_negative'))
             AND ai_sentiment_object:categories[0]:sentiment::VARCHAR IS NOT NULL
            THEN 'sentiment_mismatch'
        END,
        CASE 
            WHEN ai_sentiment_object:categories[0]:sentiment::VARCHAR = 'mixed' 
           THEN 'mixed_sentiment'
        END
    ) AS flag_reasons,
    CURRENT_TIMESTAMP() AS validation_timestamp
FROM ai_analysis;

Select * from chat_validation_results LIMIT 5;

### Analyzing Validation Results

Let's see how many chats were flagged for discrepancies and what types of issues were found.

In [ ]:
SELECT 
    is_flagged,
    COUNT(*) as chat_count,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 1) as percentage
FROM chat_validation_results
GROUP BY is_flagged;

---
## Part 4: Cross-Validating Tickets and Chats

The final analysis uses `AI_COMPLETE` for a complex task: **semantic comparison** between support tickets and their linked chat conversations.

**Why this matters:**
- Tickets are often created *after* chats, sometimes by different people
- Information can be lost or misrepresented in translation
- Identifying misalignments helps improve data quality and customer outcomes

**What we check:**
- Is the core issue the same?
- Is the product/service consistent?
- Are categories aligned?

In [ ]:
CREATE OR REPLACE TABLE ticket_chat_alignment AS
WITH ticket_chat_data AS (
    SELECT 
        t.ticket_id,
        t.ticket_number,
        t.customer_name AS ticket_customer_name,
        t.subject AS ticket_subject,
        t.description AS ticket_description,
        t.self_reported_category AS ticket_category,
        t.priority AS ticket_priority,
        t.status AS ticket_status,
        t.product_affected AS ticket_product,
        cv.chat_id,
        cv.customer_name AS chat_customer_name,
        cv.self_reported_category AS chat_category,
        cv.self_reported_sentiment AS chat_sentiment,
        c.product_mentioned AS chat_product,
        cv.full_conversation
    FROM support_tickets t
    INNER JOIN chat_validation_results cv ON t.ticket_id = cv.ticket_id
    INNER JOIN chat_logs c ON cv.chat_id = c.chat_id
),
ai_responses AS (
    SELECT 
        ticket_id,
        ticket_number,
        chat_id,
        ticket_customer_name,
        chat_customer_name,
        ticket_category,
        chat_category,
        ticket_product,
        chat_product,
        ticket_subject,
        ticket_description,
        full_conversation,
        AI_COMPLETE(
            model => 'claude-sonnet-4-5',
            prompt => 'Compare the ticket and chat to determine if they discuss the same issue.\n\n' ||
                'TICKET:\n' ||
                'Subject: ' || ticket_subject || '\n' ||
                'Description: ' || ticket_description || '\n' ||
                'Category: ' || ticket_category || '\n' ||
                'Product: ' || COALESCE(ticket_product, 'Not specified') || '\n\n' ||
                'CHAT:\n' ||
                'Conversation: ' || full_conversation || '\n' ||
                'Category: ' || chat_category || '\n' ||
                'Product: ' || COALESCE(chat_product, 'Not mentioned') || '\n\n' ||
                'Analyze if the ticket and chat are about the SAME core issue. Consider:\n' ||
                '1. Is the main problem/request the same?\n' ||
                '2. Is the product/service the same?\n' ||
                '3. Is the technical issue consistent?\n\n' ||
                'Respond ONLY with valid JSON (no markdown formatting):\n' ||
                '{"alignment":"aligned|misaligned|partial","confidence":"high|medium|low",' ||
                '"reason":"brief explanation","severity":"critical|moderate|minor"}',
            model_parameters => OBJECT_CONSTRUCT('temperature', 0.1)
        ) AS ai_alignment_response_raw,
        REGEXP_REPLACE(
            REGEXP_REPLACE(ai_alignment_response_raw, '^```json\\s*', ''),
            '\\s*```$', 
            ''
        ) AS ai_alignment_response
    FROM ticket_chat_data
)
SELECT 
    ticket_id,
    ticket_number,
    chat_id,
    ticket_customer_name,
    chat_customer_name,
    ticket_category,
    chat_category,
    ticket_product,
    chat_product,
    ticket_subject,
    ticket_description,
    full_conversation,
    ai_alignment_response,
    PARSE_JSON(ai_alignment_response):alignment::VARCHAR AS alignment_status,
    PARSE_JSON(ai_alignment_response):confidence::VARCHAR AS alignment_confidence,
    PARSE_JSON(ai_alignment_response):reason::VARCHAR AS alignment_reason,
    PARSE_JSON(ai_alignment_response):severity::VARCHAR AS misalignment_severity,
    CASE 
        WHEN ticket_category = chat_category THEN FALSE
        ELSE TRUE
    END AS category_mismatch_flag,
    CASE 
        WHEN ticket_product IS NOT NULL 
         AND chat_product IS NOT NULL
         AND LOWER(ticket_product) != LOWER(chat_product) 
        THEN TRUE
        ELSE FALSE
    END AS product_mismatch_flag,
    CASE 
        WHEN PARSE_JSON(ai_alignment_response):alignment::VARCHAR = 'misaligned' THEN TRUE
        WHEN PARSE_JSON(ai_alignment_response):alignment::VARCHAR = 'partial' 
         AND PARSE_JSON(ai_alignment_response):severity::VARCHAR IN ('critical', 'moderate') THEN TRUE
        WHEN ticket_category != chat_category THEN TRUE
        WHEN ticket_product IS NOT NULL 
         AND chat_product IS NOT NULL
         AND LOWER(ticket_product) != LOWER(chat_product) THEN TRUE
        ELSE FALSE
    END AS is_flagged,
    ARRAY_CONSTRUCT_COMPACT(
        CASE 
            WHEN PARSE_JSON(ai_alignment_response):alignment::VARCHAR = 'misaligned' 
            THEN 'issue_misalignment'
        END,
        CASE 
            WHEN PARSE_JSON(ai_alignment_response):alignment::VARCHAR = 'partial' 
            THEN 'partial_alignment'
        END,
        CASE 
            WHEN ticket_category != chat_category 
            THEN 'category_mismatch'
        END,
        CASE 
            WHEN ticket_product IS NOT NULL 
             AND chat_product IS NOT NULL
             AND LOWER(ticket_product) != LOWER(chat_product)
            THEN 'product_mismatch'
        END
    ) AS flag_reasons,
    CURRENT_TIMESTAMP() AS validation_timestamp
FROM ai_responses;

SELECT * FROM ticket_chat_alignment LIMIT 5;

### Alignment Results Summary

Let's see how well tickets and chats align, and what types of discrepancies were found.

In [ ]:
SELECT 
    alignment_status,
    alignment_confidence,
    COUNT(*) as count
FROM ticket_chat_alignment
GROUP BY alignment_status, alignment_confidence
ORDER BY count DESC;

---
## Conclusion

You've built a complete multimodal customer service analytics system using Snowflake Cortex AI functions!

### Key Takeaways

| Function | Best For |
|----------|----------|
| `AI_TRANSCRIBE` | Converting audio/video to searchable text |
| `AI_TRANSLATE` | Multilingual data normalization |
| `AI_SENTIMENT` | Customer satisfaction monitoring |
| `AI_CLASSIFY` | Zero-shot categorization with custom labels |
| `AI_COMPLETE` | Flexible LLM tasks (summaries, comparisons, generation) |
| `AI_PARSE_DOCUMENT` | Structured extraction from PDFs |
| `AI_EXTRACT` | Pulling specific fields from unstructured text |

### Next Steps
- **Scale up**: Process your own audio files and documents
- **Customize categories**: Modify AI_CLASSIFY categories for your business
- **Build dashboards**: Use the results tables for Streamlit or BI tools
- **Automate**: Schedule these queries with Tasks for continuous processing

### Resources
- [Cortex AI Functions Documentation](https://docs.snowflake.com/en/sql-reference/functions-ai)
- [Snowflake Notebooks Guide](https://docs.snowflake.com/en/user-guide/ui-snowsight/notebooks)